# EDA on PaySim Dataset

1. characterize the fraud typology 
2. show static rules are insufficient
3. quantify *why* the flagged columns are unusable

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

from fraud_detection.config import load_params, PROJECT_ROOT

params = load_params()
RAW = PROJECT_ROOT / params["paths"]["raw"]

# 6.3M rows. Downcast floats 64->32 to roughly halve the heavy columns.
dtypes = {
    "step": "int32",
    "type": "category",
    "amount": "float32",
    "nameOrig": "string",
    "oldbalanceOrg": "float32",
    "newbalanceOrig": "float32",
    "nameDest": "string",
    "oldbalanceDest": "float32",
    "newbalanceDest": "float32",
    "isFraud": "int8",
    "isFlaggedFraud": "int8",
}
df = pd.read_csv(RAW, dtype=dtypes)
print(f"{len(df):,} rows, {df.memory_usage(deep=True).sum() / 1e9:.2f} GB in RAM")
df.head()

6,362,620 rows, 1.03 GB in RAM


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.639648,C1231006815,170136.0,160296.359375,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.280029,C1666544295,21249.0,19384.720703,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.000000,C1305486145,181.0,0.000000,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.000000,C840083671,181.0,0.000000,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.139648,C2048537720,41554.0,29885.859375,M1230701703,0.0,0.0,0,0


In [7]:
print(f"Fraud rate (overall): {df['isFraud'].mean():.4%}")

fraud_by_type = df.groupby("type", observed=True)["isFraud"].agg(
    fraud_count="sum", fraud_rate="mean", total="count"
)
print(fraud_by_type)

Fraud rate (overall): 0.1291%
          fraud_count  fraud_rate    total
type                                      
CASH_IN             0    0.000000  1399284
CASH_OUT         4116    0.001840  2237500
DEBIT               0    0.000000    41432
PAYMENT             0    0.000000  2151495
TRANSFER         4097    0.007688   532909


In [8]:
print(df["isFlaggedFraud"].sum(), "transactions flagged by the built-in rule")
print(
    pd.crosstab(
        df["isFlaggedFraud"], df["isFraud"], rownames=["flagged"], colnames=["actual_fraud"]
    )
)

16 transactions flagged by the built-in rule
actual_fraud        0     1
flagged                    
0             6354407  8197
1                   0    16


In [ ]:
sub = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()
print(f"Modelled subset: {len(sub):,} rows, fraud rate {sub['isFraud'].mean():.4%}\n")

# The cancellation artifact: for fraud rows the origin shows a full-drain signature.
for label, mask in [("fraud", sub.isFraud == 1), ("legit", sub.isFraud == 0)]:
    zero_new = (sub.loc[mask, "newbalanceOrig"] == 0).mean()
    full_drn = np.isclose(sub.loc[mask, "amount"], sub.loc[mask, "oldbalanceOrg"]).mean()
    print(
        f"{label:>5}:  newbalanceOrig==0 in {zero_new:5.1%}   |   amount==oldbalanceOrg in {full_drn:5.1%}"
    )

# Quantify leakage: a single hand-built balance flag, zero modelling.
full_drain_flag = np.isclose(sub["amount"], sub["oldbalanceOrg"]).astype(int)
auc = roc_auc_score(sub["isFraud"], full_drain_flag)
print(f"\nAUC of ONE hand-built balance flag (no model): {auc:.3f}")
print("=> Confirms the data card: these columns trivially encode the label and must be excluded.")

Modelled subset: 2,770,409 rows, fraud rate 0.2965%

fraud:  newbalanceOrig==0 in 98.1%   |   amount==oldbalanceOrg in 97.8%
legit:  newbalanceOrig==0 in 90.1%   |   amount==oldbalanceOrg in  0.0%

AUC of ONE hand-built balance flag (no model): 0.989
=> Confirms the data card: these columns trivially encode the label and must be excluded.


In [ ]:
# Downsample
n_legit = params["sample"]["n_legit"]
seed = params["sample"]["random_state"]

fraud = sub[sub.isFraud == 1]
legit = sub[sub.isFraud == 0].sample(n=n_legit, random_state=seed)
sample = pd.concat([fraud, legit]).sample(frac=1, random_state=seed)[  # shuffle
    df.columns
]  # original schema only

out = PROJECT_ROOT / params["paths"]["sample"]
sample.to_csv(out, index=False, float_format="%.2f")  # currency = 2dp, keeps file clean & small
print(f"Wrote {len(sample):,} rows -> {out}")
print(f"  fraud: {sample.isFraud.sum():,} ({sample.isFraud.mean():.3%})")

Wrote 208,213 rows -> /Users/haizad/Documents/projects/fraud-detection-demo/data/01_raw/paysim_sample.csv
  fraud: 8,213 (3.945%)
